# Cross-Market Arbitrage: KXNBAGAME vs KXNBASPREAD

Explores temporal price divergences between the win/loss market (KXNBAGAME) and the
spread market at the 1.5-point strike (KXNBASPREAD). These should trade within ~$0.02
of each other since P(win by 2+) ≈ P(win) minus the rare 1-point win.

**Key finding:** 44% of 30s bins show >3c divergence across 35 game pairs. However,
the backtest has a critical flaw: it does not simulate actual settlement outcomes
(1-point wins break the arb), and SPREAD@1.5 liquidity is very thin (43-74 trades/game).

In [ ]:
import json
import gzip
import re
import gc
from concurrent.futures import ThreadPoolExecutor, as_completed
from datetime import datetime, timezone, timedelta

import boto3
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

s3 = boto3.client("s3")
S3_BUCKET = "prediction-markets-data"

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 120)

## 1. Cross-Market Arbitrage: KXNBASPREAD → Implied P(Win) vs KXNBAGAME

### The two contracts

Both markets are binary ($0 or $1 at settlement). For a given game, say **LAL vs DAL**:

| Contract | Pays $1 if... | Probability of payoff |
|---|---|---|
| `KXNBAGAME-LAL` (YES) | LAL wins the game | `P(LAL wins)` |
| `KXNBASPREAD-LAL-1.5` (YES) | LAL wins by **2 or more** points | `P(LAL wins by ≥2)` |

The spread contract is *strictly harder to win* than the game contract, because it excludes the case "LAL wins by exactly 1 point." So by basic probability:

```
P(LAL wins)  =  P(LAL wins by ≥2)  +  P(LAL wins by exactly 1)
```

Kalshi prices are (roughly) probabilities in cents, so in a fair market:

```
price(GAME-LAL)  =  price(SPREAD-LAL-1.5)  +  P(1-point win)
                 ≈  price(SPREAD-LAL-1.5)  +  2–3¢
```

That **2–3¢ gap is the only structurally justifiable price difference** between the two. In NBA history, roughly 2–3% of games are decided by exactly 1 point, so if GAME trades 2–3¢ above SPREAD@1.5, everyone is priced fairly.

### Where the arb comes from

Now suppose at some moment during the game, the quotes look like this:

```
GAME-LAL         : 0.68  ← implied P(win) = 68%
SPREAD-LAL-1.5   : 0.63  ← implied P(win by ≥2) = 63%
gap              : 0.05  ← 5¢
```

The gap is 5¢, but the *true* gap (the 1-point-win probability) is only ~2¢. So **one of these two prices is wrong by ~3¢**. We don't need to know *which one*. The arb doesn't care — it just trades the spread between them.

### The trade (when gap is too wide)

Gap of 5¢ means GAME is too expensive relative to SPREAD, OR SPREAD is too cheap relative to GAME. Either way:

- **Sell GAME-LAL YES @ 0.68** (or equivalently, buy NO @ 0.32)
- **Buy SPREAD-LAL-1.5 YES @ 0.63**

Now look at what happens at settlement in each possible outcome:

| Game outcome | GAME YES pays | SPREAD YES pays | Net P&L per pair |
|---|---|---|---|
| LAL wins by ≥2 | $1 (we owe it) | $1 (we collect it) | `-1 + 1 + (0.68 - 0.63) = +$0.05` |
| LAL wins by exactly 1 | $1 (we owe it) | $0 | `-1 + 0 + (0.68 - 0.63) = -$0.95` |
| LAL loses | $0 | $0 | `0 + 0 + (0.68 - 0.63) = +$0.05` |

So in **two of three outcome categories we make 5¢**, and in the (rare) 1-point-win case we lose 95¢. Expected value:

```
E[P&L] = 0.05 × P(not a 1-point LAL win) - 0.95 × P(1-point LAL win)
       ≈ 0.05 × 0.98           -  0.95 × 0.02
       ≈ +0.049                -  0.019
       ≈ +0.030 per contract pair
```

Positive EV of ~3¢ per contract, which is exactly the mispricing we identified. Scale up → repeat across many games → collect the edge.

### Why this isn't free money

- **It's not a true arb, it's a statistical arb.** There *is* a path where you lose (the 1-point game). The edge only materializes over many trades. Sequence one bad 1-point game before you've diversified and you're underwater.
- **Fees and the spread eat you alive.** Kalshi fees plus the bid-ask round trip can easily cost 2–4¢ per pair. You need the gap to be materially *wider* than both the 2¢ fundamental gap **and** transaction costs before entering.
- **You need to trade both legs.** If you only get filled on one leg, you have directional exposure instead of arb — now you're just betting on the game.

### Temporal, not structural

At **settlement**, GAME and SPREAD@1.5 always reconcile (to within the 1-point probability) — that's the law of the payoffs above. So any wide gap you see during a game is a *temporary* dislocation: one market saw new information (a scoring run, an injury, a lead change) and repriced faster than the other. The mispricing closes on its own, usually within seconds to minutes.

The strategy is therefore a **convergence trade**: enter when the gap opens wider than ~fundamental + fees, exit when it closes (or hold to settlement). You are not discovering a permanent flaw in contract design — you are supplying liquidity between two correlated pools that temporarily disagree.

### What this notebook section measures

The cells below quantify: (1) how often the gap exceeds a tradeable threshold, (2) how long it persists before closing, (3) what the realized P&L looks like net of realistic fills and fees. Those numbers determine whether this is a real strategy or a textbook curiosity.


In [ ]:
# Load all KXNBAGAME and KXNBASPREAD markets
obj = s3.get_object(Bucket=S3_BUCKET, Key="kalshi/historical_markets/KXNBAGAME.json")
game_markets = pd.DataFrame(json.loads(obj["Body"].read()))
print(f"KXNBAGAME: {len(game_markets)} markets")

obj = s3.get_object(Bucket=S3_BUCKET, Key="kalshi/historical_markets/KXNBASPREAD.json")
spread_markets = pd.DataFrame(json.loads(obj["Body"].read()))
print(f"KXNBASPREAD: {len(spread_markets)} markets")

# Parse event_ticker to extract game info
# KXNBAGAME: event_ticker = "KXNBAGAME-25DEC25MINDEN", ticker = "KXNBAGAME-25DEC25MINDEN-DEN"
# KXNBASPREAD: event_ticker = "KXNBASPREAD-25DEC25MINDEN", ticker = "KXNBASPREAD-25DEC25MINDEN-DEN7"

# Extract the game key (date + teams) from event_ticker
def game_key_from_event(event_ticker, prefix):
    """Extract 'YYMMMDDAWYHOM' from event_ticker, e.g., '25DEC25MINDEN'"""
    return event_ticker.replace(prefix + "-", "")

game_markets["game_key"] = game_markets["event_ticker"].apply(lambda x: game_key_from_event(x, "KXNBAGAME"))
spread_markets["game_key"] = spread_markets["event_ticker"].apply(lambda x: game_key_from_event(x, "KXNBASPREAD"))

# Find games that exist in both markets
common_games = set(game_markets["game_key"]) & set(spread_markets["game_key"])
print(f"\nGames in both GAME and SPREAD markets: {len(common_games)}")

# Parse spread strike from ticker
# e.g., "KXNBASPREAD-25DEC25MINDEN-DEN7" → team=DEN, strike=7 → "DEN wins by >7.5"
def parse_spread_ticker(ticker):
    parts = ticker.split("-")
    team_strike = parts[-1]  # e.g., "DEN7" or "MIN3"
    match = re.match(r"([A-Z]+)(\d+)", team_strike)
    if match:
        team = match.group(1)
        strike = int(match.group(2)) + 0.5  # strikes are X.5
        return team, strike
    return None, None

spread_markets[["strike_team", "strike_val"]] = spread_markets["ticker"].apply(
    lambda x: pd.Series(parse_spread_ticker(x))
)
print(f"\nSpread market strike distribution:")
print(spread_markets["strike_val"].describe())

In [ ]:
# For each game, compare KXNBAGAME last_price vs KXNBASPREAD lowest-strike implied P(win)
# The spread market "Team wins by >1.5" ≈ P(team wins) in NBA (no ties, 1-point wins are ~2%)

# Get KXNBAGAME prices per game (home team market)
# Each game has 2 markets (home + away team), prices should sum to ~$1.00
game_by_event = game_markets.groupby("event_ticker").apply(
    lambda g: pd.Series({
        "game_key": g["game_key"].iloc[0],
        "tickers": list(g["ticker"]),
        "prices": list(g["last_price_dollars"].astype(float)),
        "expiration_values": list(g["expiration_value"]),
    })
).reset_index()

print(f"KXNBAGAME events: {len(game_by_event)}")
print(f"Price sum distribution (should be ~1.0):")
game_by_event["price_sum"] = game_by_event["prices"].apply(sum)
print(game_by_event["price_sum"].describe())

# For spread: find the lowest strike for each team in each game
# "Team wins by >1.5" is the closest to "team wins"
spread_low = spread_markets[spread_markets["strike_val"] <= 2.5].copy()
print(f"\nSpread markets with strike <= 2.5: {len(spread_low)}")
print(spread_low.groupby("game_key").size().describe())

In [ ]:
# Load trades for paired GAME + SPREAD markets and compare prices over time
# Pick games with high volume for clearer signal

# Strategy: for each game, load trades from KXNBAGAME-{key}-{team} and KXNBASPREAD-{key}-{team}1
# (the "wins by >1.5" strike), align on time, measure divergence

def load_trades(ticker):
    """Load historical trades for a ticker from S3."""
    try:
        obj = s3.get_object(Bucket=S3_BUCKET, Key=f"kalshi/historical_trades/{ticker}.json")
        trades = json.loads(obj["Body"].read())
        df = pd.DataFrame(trades)
        if not df.empty:
            df["created_time"] = pd.to_datetime(df["created_time"])
            df["yes_price"] = df["yes_price_dollars"].astype(float)
        return df
    except Exception:
        return pd.DataFrame()

# Find games where both GAME and SPREAD "wins by 1.5" exist
# Build pairs: (game_ticker, spread_ticker_1.5) for each team in each game
pairs = []
for game_key in list(common_games)[:200]:  # sample 200 games
    # Get game market tickers for this game
    gm = game_markets[game_markets["game_key"] == game_key]
    # Get spread markets with strike 1.5 for this game
    sm = spread_markets[(spread_markets["game_key"] == game_key) & (spread_markets["strike_val"] == 1.5)]
    
    for _, grow in gm.iterrows():
        # Extract team from game ticker (last segment)
        game_team = grow["ticker"].split("-")[-1]
        # Find matching spread ticker for same team
        matching_spread = sm[sm["strike_team"] == game_team]
        if not matching_spread.empty:
            pairs.append({
                "game_key": game_key,
                "team": game_team,
                "game_ticker": grow["ticker"],
                "spread_ticker": matching_spread.iloc[0]["ticker"],
                "game_last_price": float(grow["last_price_dollars"]),
                "spread_last_price": float(matching_spread.iloc[0]["last_price_dollars"]),
            })

pairs_df = pd.DataFrame(pairs)
print(f"Found {len(pairs_df)} GAME↔SPREAD(1.5) pairs across {pairs_df['game_key'].nunique()} games")

# Compare last prices
pairs_df["price_diff"] = pairs_df["game_last_price"] - pairs_df["spread_last_price"]
print(f"\nLast price difference (GAME - SPREAD@1.5):")
print(pairs_df["price_diff"].describe())
print(f"\n|diff| > 0.03: {(pairs_df['price_diff'].abs() > 0.03).sum()} pairs")
print(f"|diff| > 0.05: {(pairs_df['price_diff'].abs() > 0.05).sum()} pairs")

In [ ]:
# Time-series comparison: load trades for top divergent pairs and measure intra-game spread
# Pick the pairs with largest |price_diff| at settlement

top_pairs = pairs_df.nlargest(20, "price_diff", keep="first").head(10)
print("Top 10 divergent pairs (GAME price > SPREAD@1.5 price):")
print(top_pairs[["game_key", "team", "game_last_price", "spread_last_price", "price_diff"]].to_string())

# Load intra-game trade series for a few high-divergence pairs
def load_pair_trades(row):
    game_trades = load_trades(row["game_ticker"])
    spread_trades = load_trades(row["spread_ticker"])
    return game_trades, spread_trades

# Load trades for top 5 pairs
print("\n\nLoading trade time series for top divergent pairs...")
pair_series = []
for idx, row in top_pairs.head(5).iterrows():
    gt, st = load_pair_trades(row)
    if not gt.empty and not st.empty:
        pair_series.append({
            "game_key": row["game_key"],
            "team": row["team"],
            "game_trades": gt,
            "spread_trades": st,
            "game_ticker": row["game_ticker"],
            "spread_ticker": row["spread_ticker"],
        })
        print(f"  {row['game_key']} {row['team']}: {len(gt)} game trades, {len(st)} spread trades")

print(f"\nLoaded {len(pair_series)} pairs with trade data")

In [ ]:
# Plot GAME vs SPREAD@1.5 price over time for divergent pairs
fig, axes = plt.subplots(min(len(pair_series), 3), 1, figsize=(14, 4*min(len(pair_series), 3)), squeeze=False)

for i, ps in enumerate(pair_series[:3]):
    ax = axes[i, 0]
    gt = ps["game_trades"].sort_values("created_time")
    st = ps["spread_trades"].sort_values("created_time")
    
    ax.step(gt["created_time"], gt["yes_price"], where="post", label=f"GAME {ps['team']}", alpha=0.8)
    ax.step(st["created_time"], st["yes_price"], where="post", label=f"SPREAD@1.5 {ps['team']}", alpha=0.8, linestyle="--")
    ax.set_title(f"{ps['game_key']} — {ps['team']}")
    ax.set_ylabel("Yes price ($)")
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.suptitle("KXNBAGAME vs KXNBASPREAD@1.5 — should track closely", y=1.01, fontsize=13)
plt.show()

In [ ]:
# Systematic cross-market divergence measurement
# For each pair, align trades on 30-second bins and measure price gap over time

def measure_divergence_over_time(game_trades, spread_trades, freq="30s"):
    """Resample both trade series to common frequency, measure gap."""
    if game_trades.empty or spread_trades.empty:
        return pd.DataFrame()
    
    gt = game_trades.set_index("created_time")["yes_price"].resample(freq).last().ffill()
    st = spread_trades.set_index("created_time")["yes_price"].resample(freq).last().ffill()
    
    # Align on common timestamps
    combined = pd.DataFrame({"game_price": gt, "spread_price": st}).dropna()
    if combined.empty:
        return combined
    
    combined["divergence"] = combined["game_price"] - combined["spread_price"]
    return combined

# Measure across all loaded pairs
all_divergences = []
for ps in pair_series:
    div = measure_divergence_over_time(ps["game_trades"], ps["spread_trades"])
    if not div.empty:
        all_divergences.append(div["divergence"])
        
if all_divergences:
    all_div = pd.concat(all_divergences)
    print(f"Cross-market divergence (GAME - SPREAD@1.5), 30s bins:")
    print(f"  Observations: {len(all_div)}")
    print(f"  Mean:   {all_div.mean():.4f}")
    print(f"  Std:    {all_div.std():.4f}")
    print(f"  Median: {all_div.median():.4f}")
    print(f"  |div| > $0.02: {(all_div.abs() > 0.02).mean():.1%} of observations")
    print(f"  |div| > $0.03: {(all_div.abs() > 0.03).mean():.1%} of observations")
    print(f"  |div| > $0.05: {(all_div.abs() > 0.05).mean():.1%} of observations")

# Now do the same at scale — load many more pairs
print("\n\nScaling up: loading trades for 50 random common games...")
import random
sample_games = random.sample(list(common_games), min(50, len(common_games)))

scale_divergences = []
loaded = 0
for game_key in sample_games:
    gm = game_markets[game_markets["game_key"] == game_key]
    sm = spread_markets[(spread_markets["game_key"] == game_key) & (spread_markets["strike_val"] == 1.5)]
    
    for _, grow in gm.iterrows():
        game_team = grow["ticker"].split("-")[-1]
        matching_spread = sm[sm["strike_team"] == game_team]
        if not matching_spread.empty:
            gt = load_trades(grow["ticker"])
            st = load_trades(matching_spread.iloc[0]["ticker"])
            div = measure_divergence_over_time(gt, st)
            if not div.empty:
                scale_divergences.append(div["divergence"])
                loaded += 1
    if loaded >= 50:
        break

if scale_divergences:
    all_div_scale = pd.concat(scale_divergences)
    print(f"\n  Loaded {loaded} pairs, {len(all_div_scale)} time-aligned observations")
    print(f"  Mean divergence:  {all_div_scale.mean():.4f}")
    print(f"  Std:              {all_div_scale.std():.4f}")
    print(f"  |div| > $0.02:   {(all_div_scale.abs() > 0.02).mean():.1%}")
    print(f"  |div| > $0.03:   {(all_div_scale.abs() > 0.03).mean():.1%}")
    print(f"  |div| > $0.05:   {(all_div_scale.abs() > 0.05).mean():.1%}")
    print(f"  Max |div|:       ${all_div_scale.abs().max():.3f}")

In [ ]:
# Cross-market arb backtest
# When |GAME - SPREAD@1.5| > threshold, buy the cheap one and sell the expensive one
# Both settle to same outcome (team wins or not), so guaranteed convergence at expiry

ARB_THRESHOLD = 0.03  # only trade when gap > 3 cents
ARB_FEE = 0.02  # fee per leg (2 legs = $0.04 total)

arb_trades = []
for div_series in scale_divergences:
    for ts, div_val in div_series.items():
        if abs(div_val) > ARB_THRESHOLD:
            # If GAME > SPREAD: sell GAME, buy SPREAD (both settle same)
            # Profit = |divergence| - 2*fee at expiry
            pnl = abs(div_val) - 2 * ARB_FEE
            arb_trades.append({
                "time": ts,
                "divergence": div_val,
                "direction": "sell_game" if div_val > 0 else "buy_game",
                "pnl": pnl,
            })

if arb_trades:
    arb_df = pd.DataFrame(arb_trades)
    print(f"CROSS-MARKET ARB BACKTEST (threshold={ARB_THRESHOLD}, fee={ARB_FEE}/leg)")
    print("=" * 70)
    print(f"  Signals: {len(arb_df)}")
    print(f"  Win rate: {(arb_df['pnl'] > 0).mean():.1%}")
    print(f"  Mean PnL/trade: ${arb_df['pnl'].mean():.4f}")
    print(f"  Total PnL: ${arb_df['pnl'].sum():.2f}")
    print(f"  Mean |divergence|: ${arb_df['divergence'].abs().mean():.4f}")
    print(f"  Sharpe: {arb_df['pnl'].mean() / arb_df['pnl'].std():.2f}" if arb_df['pnl'].std() > 0 else "")
    print(f"\n  By direction:")
    for d, grp in arb_df.groupby("direction"):
        print(f"    {d}: {len(grp)} trades, mean PnL ${grp['pnl'].mean():.4f}")
    
    # NOTE: This assumes we can execute at the observed divergence (no slippage).
    # In reality, we'd need to cross the spread on both legs simultaneously.
    # The order book analysis below tells us if this is feasible.
else:
    print("No arb signals found at this threshold — markets are well-linked.")